# Практикум 2. Очистка текста, нормализация и BPE

В лаборатории 2 вы включали шаги очистки и смотрели, как BPE склеивает буквы в куски слов. Здесь то же самое в коде на Python:

1. разберёмся, почему одинаковые на вид строки бывают разными (Unicode, NFC, похожие буквы разных алфавитов);
2. соберём конвейер нормализации из маленьких функций и посчитаем, как падает число типов;
3. напишем обучение BPE с нуля и напечатаем каждое слияние;
4. применим слияния к новым словам и сравним с библиотекой `tokenizers`;
5. посчитаем фертильность (сколько токенов приходится на слово) на корпусе побольше.

*Практикум к лаборатории 2 курса «Алгоритмы NLP». Запускайте ячейки сверху вниз: Shift+Enter или «Среда выполнения → Выполнить всё». Чтобы сохранить свои ответы, сделайте «Файл → Сохранить копию на Диске».*

## 1. Unicode: одинаковые на вид, разные внутри

Компьютер хранит не буквы, а коды символов Unicode. Букву «ё» можно записать одним кодом (U+0451) или двумя: «е» + комбинируемое двоеточие (U+0308). На экране разницы нет, но строки не равны, и словарь посчитает их разными словами. **Нормальная форма NFC** склеивает такие пары в один символ. Вторая ловушка: латинские «o», «a», «c» выглядят как русские, и слово со смесью алфавитов тоже станет отдельным типом. Третья: **NFKC** заменяет «совместимые» символы (², №, лигатуры) обычными.

In [ ]:
import re
import unicodedata

a = "ёлка"            # ё одним символом
b = "е\u0308лка"      # е + комбинируемое двоеточие
print(a, b, "равны?", a == b, "| длины:", len(a), len(b))
for s in (a, b):
    print("   ", [f"U+{ord(c):04X} {unicodedata.name(c)}" for c in s[:2]])
b_nfc = unicodedata.normalize("NFC", b)
print("после NFC равны?", b_nfc == a, "| длина:", len(b_nfc))

def scripts(word):
    # из каких алфавитов буквы слова: CYRILLIC, LATIN …
    return sorted({unicodedata.name(c).split()[0] for c in word if c.isalpha()})

print()
for w in ["Москва", "Мoсква", "сахар", "caxap"]:   # во 2-м слове латинская o, в 4-м всё латиницей
    print(f"{w:<8} {scripts(w)}")

print()
s = "№ 5, 10 м², ½ стакана"
print("NFKC:", s, "→", unicodedata.normalize("NFKC", s))

Запишем это в виде функций, которые пригодятся в конвейере. Замена «ё» на «е» — обычный шаг для русского: в текстах «ё» часто не пишут, и «ёлка» с «елка» должны совпасть.

In [ ]:
def to_nfc(text):
    return unicodedata.normalize("NFC", text)

def yo_to_e(text):
    return text.replace("ё", "е").replace("Ё", "Е")

def mixed_script_words(text):
    # слова, где смешаны кириллица и латиница
    return [w for w in re.findall(r"\w+", text) if len(scripts(w)) > 1]

print(yo_to_e(to_nfc("Е\u0308лка и ёлка")))
print(mixed_script_words("Мoсква и Москва, сахар и caxap"))

## 2. Конвейер нормализации

Возьмём «грязный» текст из лаборатории. Каждый шаг очистки — функция «текст → текст», конвейер — их список. Токены, как в лаборатории, получаем простым разбиением по пробелу, поэтому два пробела подряд дают пустой токен `''`. Будем считать **токены** (все слова подряд) и **типы** (разные слова) после каждого шага.

In [ ]:
SAMPLE = "Ёлка в Москве!!!  Ёлку   поставили 31 декабря, ЁЛКА высокая. Елка — символ праздника: https://news.ru/elka-2024 . В 2024 году ёлка была выше, чем в 2023 году."
STOP = {"в", "во", "на", "и", "а", "но", "с", "со", "к", "по", "у", "о", "из", "за", "для", "это",
        "был", "была", "было", "были", "чем", "что", "как", "же", "не"}

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
STEPS = [
    ("нижний регистр",       lambda t: t.lower()),
    ("ё → е",                yo_to_e),
    ("удаление ссылок",      lambda t: re.sub(r"https?://\S+", "", t)),
    ("удаление пунктуации",  lambda t: re.sub(r"[^\w\s]", "", t)),
    ("удаление цифр",        lambda t: re.sub(r"\d+", "", t)),
    ("схлопывание пробелов", lambda t: re.sub(r" +", " ", t).strip()),
    ("удаление стоп-слов",   lambda t: " ".join(w for w in t.split(" ") if w not in STOP)),
]

def run_steps(text, names):
    fns = dict(STEPS)
    for name in names:
        text = fns[name](text)
    return text

def stats(text):
    toks = text.split(" ")
    return len(toks), len(set(toks))

text = SAMPLE
rows = [("исходный текст",) + stats(text)]
for name, fn in STEPS:
    text = fn(text)
    rows.append((name,) + stats(text))
table = pd.DataFrame(rows, columns=["этап", "токенов", "типов"])
print(table.to_string(index=False))
print()
print("Токены после всех шагов:")
print(text.split(" "))
print()
# Порядок важен: если удалить пунктуацию раньше ссылок, от ссылки останется «слово»
good = run_steps(SAMPLE, ["нижний регистр", "ё → е", "удаление ссылок", "удаление пунктуации"])
bad = run_steps(SAMPLE, ["нижний регистр", "ё → е", "удаление пунктуации", "удаление ссылок"])
print("Пунктуация раньше ссылок, лишние типы:", sorted(set(bad.split()) - set(good.split())))

plt.figure(figsize=(8, 3))
plt.bar(range(len(table)), table["типов"], color="#2B55C9")
plt.xticks(range(len(table)), table["этап"], rotation=35, ha="right")
plt.title("Число типов после каждого шага")
plt.tight_layout()
plt.show()

## 3. BPE с нуля: обучение

BPE (Byte Pair Encoding) начинает со словаря из отдельных букв и маркера конца слова `</w>`. На каждом шаге он считает все пары соседних символов (с учётом частот слов) и склеивает самую частую:

$$(a, b)^* = \arg\max_{(a,b)} \sum_{w} f(w)\cdot \mathrm{count}_w(a, b), \qquad |V| = |\text{алфавит}| + k.$$

Корпус тот же, что в лаборатории: слово и его частота. При равных частотах выигрывает пара, встреченная раньше (так же в лаборатории).

In [ ]:
corpus = [("низкий", 5), ("низкая", 2), ("новый", 6), ("новая", 3), ("новейший", 3), ("широкий", 4),
          ("широкая", 2), ("высокий", 3), ("высокая", 2), ("старый", 4), ("старая", 2), ("старейший", 2)]
new_words = ["новейшая", "старейшая", "высокие", "крупнейший", "новый"]

In [ ]:
from collections import Counter

def symbols(word):
    return tuple(word) + ("</w>",)

def pair_counts(vocab):
    counts = Counter()
    for word, freq in vocab.items():
        for a, b in zip(word, word[1:]):
            counts[a, b] += freq
    return counts

def merge_word(word, pair):
    out, i = [], 0
    while i < len(word):
        if i + 1 < len(word) and (word[i], word[i + 1]) == pair:
            out.append(word[i] + word[i + 1])
            i += 2
        else:
            out.append(word[i])
            i += 1
    return tuple(out)

def train_bpe(corpus, num_merges=None, verbose=0):
    vocab = {symbols(w): f for w, f in corpus}
    alphabet = sorted({s for w in vocab for s in w})
    n_words = sum(f for _, f in corpus)
    avg_len = lambda: sum(len(w) * f for w, f in vocab.items()) / n_words
    merges, history = [], [(0, len(alphabet), avg_len())]
    while num_merges is None or len(merges) < num_merges:
        counts = pair_counts(vocab)
        if not counts:
            break
        best = max(counts, key=counts.get)        # при равенстве частот побеждает пара, встреченная раньше
        if counts[best] < 2:
            break
        if len(merges) < verbose:
            top = ", ".join(f"{a}+{b}:{c}" for (a, b), c in counts.most_common(4))
            print(f"шаг {len(merges) + 1:>2}. частые пары: {top}")
            print(f"     слияние «{best[0]}» + «{best[1]}» → «{best[0] + best[1]}», словарь = {len(alphabet) + len(merges) + 1}")
        vocab = {merge_word(w, best): f for w, f in vocab.items()}
        merges.append(best)
        history.append((len(merges), len(alphabet) + len(merges), avg_len()))
    return merges, alphabet, vocab, history

print("Начальный алфавит:", " ".join(sorted({s for w, _ in corpus for s in symbols(w)})))
print()
merges, alphabet, seg_vocab, history = train_bpe(corpus, num_merges=8, verbose=8)

Обучим до конца (пока есть пары, встреченные хотя бы дважды) и посмотрим на кривые, как на графике лаборатории: словарь растёт на 1 с каждым слиянием, а число токенов на слово падает всё медленнее.

In [ ]:
merges, alphabet, seg_vocab, history = train_bpe(corpus)
print(f"выучено слияний: {len(merges)}; |V| = {len(alphabet)} + {len(merges)} = {len(alphabet) + len(merges)}")
print()
print("Слова корпуса после всех слияний:")
for (w, f), seg in zip(corpus, seg_vocab):
    print(f"   {w:<12} ×{f}  →  {' | '.join(seg)}")

k_axis = [h[0] for h in history]
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].plot(k_axis, [h[1] for h in history], color="#2B55C9")
ax[0].set_title("Размер словаря"); ax[0].set_xlabel("слияния"); ax[0].set_ylabel("размер словаря")
ax[1].plot(k_axis, [h[2] for h in history], color="#17907E")
ax[1].set_title("Токенов на слово"); ax[1].set_xlabel("слияния"); ax[1].set_ylabel("токенов на слово (с </w>)")
plt.tight_layout()
plt.show()

### Применение к новым словам

Новое слово разбиваем на буквы и применяем слияния **строго в порядке обучения**. Печатаем только сработавшие слияния. Буквы, которых не было в корпусе, остаются отдельными символами: классический BPE заменил бы их на `[UNK]`.

In [ ]:
def bpe_segment(word, merges, verbose=False):
    syms = symbols(word)
    for rank, pair in enumerate(merges, 1):
        if pair in zip(syms, syms[1:]):
            syms = merge_word(syms, pair)
            if verbose:
                print(f"   №{rank:<3} {pair[0]} + {pair[1]:<8} → {' '.join(syms)}")
    return list(syms)

for w in new_words:
    unknown = sorted(set(w) - set(alphabet))
    print(w, f"({len(symbols(w))} символов)", f"  нет в алфавите: {unknown}" if unknown else "")
    seg = bpe_segment(w, merges, verbose=True)
    print("   итог:", " | ".join(seg))
    print()

## 4. То же в библиотеке `tokenizers`

Библиотека Hugging Face `tokenizers` обучает BPE на том же корпусе. Она предустановлена в Colab. Если её нет (как при запуске без интернета), ячейка просто пропускается. Сравните слияния и разбиения с нашими. В библиотеке маркер `</w>` с самого начала приклеен к последней букве, поэтому слияния «й + </w>» нет, пары считаются чуть иначе и разбиения немного отличаются (например, «ейш» библиотека не выучила), а символы, которых нет в её словаре, превращаются в `[UNK]`. Это не только новые буквы: «е» в корпусе есть, но ни разу не стоит в конце слова, поэтому символа «е</w>» в словаре нет.

In [ ]:
try:
    from tokenizers import Tokenizer, models, trainers, pre_tokenizers
    HAVE_TOKENIZERS = True
except ImportError:
    HAVE_TOKENIZERS = False
    print("Библиотека tokenizers не установлена: ячейка пропущена (в Colab она есть).")

if HAVE_TOKENIZERS:
    import json
    tok = Tokenizer(models.BPE(unk_token="[UNK]", end_of_word_suffix="</w>"))
    tok.pre_tokenizer = pre_tokenizers.WhitespaceSplit()
    trainer = trainers.BpeTrainer(vocab_size=1000, min_frequency=2, special_tokens=["[UNK]"],
                                  end_of_word_suffix="</w>", show_progress=False)
    tok.train_from_iterator([w for w, f in corpus for _ in range(f)], trainer=trainer)
    lib_merges = json.loads(tok.to_str())["model"]["merges"]
    print("слияний в библиотеке:", len(lib_merges), "| |V| =", tok.get_vocab_size())
    print("   ", [" + ".join(m) if isinstance(m, list) else m for m in lib_merges[:8]])
    print()
    for w in new_words:
        print(f"{w:<18} наш BPE: {' | '.join(bpe_segment(w, merges)):<36} tokenizers: {' | '.join(tok.encode(w).tokens)}")

### Фертильность на корпусе побольше

**Фертильность** — среднее число токенов на слово. Для словаря из целых слов она равна 1, зато каждое новое слово неизвестно (OOV). Обучим BPE на 30 предложениях и проверим на 8 других. Одиночный `</w>` не считаем: в настоящих токенизаторах он приклеен к последней букве.

Свои тексты можно загрузить через панель «Файлы» слева (например, `texts.txt`) и прочитать их с помощью `open("texts.txt", encoding="utf-8")`.

In [ ]:
train_text = """
Дети каждое утро идут в школу. В нашей школе большая библиотека. В библиотеке много новых книг.
Ученики положили книги в рюкзаки. Учитель дал нам новую книгу. Мы бережно храним наши книги.
После уроков я играл с друзьями в саду. В нашем саду растут яблони и абрикосы. Весной деревья цветут.
Мама принесла с рынка яблоки и виноград. Папа каждый день ездит на работу на автобусе.
Москва — столица России. Казань и Новгород — древние города. В наших городах строят новые школы.
Студенты изучают в университете язык и литературу. Вчера я взял в библиотеке две книги.
Одну из книг я отдал младшему брату. Брат с интересом прочитал книгу. Летом мы ездим к бабушке в деревню.
В деревне дети купаются в реке. Бабушка рассказывает нам сказки. В сказках добро всегда побеждает.
Мы положили тетради на стол. Учителя всегда помогают ученикам. В нашем классе двадцать пять учеников.
У каждого ученика есть своя тетрадь. Чтение книг расширяет знания. Книги детей лежат на столе.
После школы мы вернулись домой. Вечером мы пили чай всей семьёй.
"""
test_text = """
Библиотекари аккуратно расставили книги по полкам. Школьные библиотеки получили новейшие словари.
В садах наших друзей созрели яблоки. Ученикам раздали новые тетради. Из городов в деревни ходят автобусы.
Младшие братья любят сказки. В университетах стало больше студентов. Дайте мне одну из ваших книг.
"""

def words(text):
    return re.findall(r"[а-я]+", yo_to_e(to_nfc(text).lower()))

In [ ]:
import numpy as np

train_words, test_words = words(train_text), words(test_text)
freq = Counter(train_words)
big_corpus = list(freq.items())
big_merges, big_alphabet, _, _ = train_bpe(big_corpus)
print(f"слов в обучении: {len(train_words)}, типов: {len(freq)}; слов в тесте: {len(test_words)}")
print(f"выучено слияний: {len(big_merges)}, |V| = {len(big_alphabet) + len(big_merges)}")
print(f"доля OOV (слово целиком не встречалось): {np.mean([w not in freq for w in test_words]):.0%}")
print(f"тестовых слов с незнакомыми буквами: {[w for w in test_words if set(w) - set(big_alphabet)]}")

def fertility(word_list, merge_list):
    # одиночный </w> не считаем
    return np.mean([sum(s != "</w>" for s in bpe_segment(w, merge_list)) for w in word_list])

Ks = sorted(set([0, 10, 25, 50, 100, 150, 200, 300, len(big_merges)]) & set(range(len(big_merges) + 1)))
fert = [(K, fertility(train_words, big_merges[:K]), fertility(test_words, big_merges[:K])) for K in Ks]
print()
print(pd.DataFrame(fert, columns=["K", "фертильность (обучение)", "фертильность (тест)"]).round(2).to_string(index=False))

print()
for w in test_words[:12]:
    print(f"   {w:<20} → {' | '.join(bpe_segment(w, big_merges))}")

plt.figure(figsize=(6, 3.2))
plt.plot(Ks, [f[1] for f in fert], "o-", label="обучение")
plt.plot(Ks, [f[2] for f in fert], "s-", label="тест")
plt.axhline(1, color="gray", ls="--", label="целые слова")
plt.xlabel("число слияний K"); plt.ylabel("фертильность"); plt.title("Сколько токенов приходится на слово")
plt.legend(); plt.tight_layout(); plt.show()

### Как это выглядит на настоящей модели

Ячейка ниже работает только в Colab (нужен интернет). Она скачивает два настоящих токенизатора, без весов моделей: GPT-2 (байтовый BPE, 50 257 токенов, файл около 1,4 МБ) и XLM-RoBERTa (SentencePiece, 250 002 токена, около 9 МБ). Сравните фертильность на одной и той же фразе по-русски, по-узбекски и по-английски.

In [ ]:
samples = {
    "ru": "Учителя нашей школы подарили ученикам новые учебники.",
    "uz": "Maktabimiz oʻqituvchilari oʻquvchilarga yangi darsliklar sovgʻa qilishdi.",
    "en": "The teachers of our school gave the students new textbooks.",
}
try:
    from tokenizers import Tokenizer
    for name, model_id in [("GPT-2", "openai-community/gpt2"), ("XLM-RoBERTa", "FacebookAI/xlm-roberta-base")]:
        tk = Tokenizer.from_pretrained(model_id)
        print(name)
        for lang, text in samples.items():
            enc = tk.encode(text, add_special_tokens=False)
            n = len(text.split())
            print(f"   {lang}: {len(enc.tokens):3d} токенов на {n} слов = {len(enc.tokens) / n:.2f}")
        if "xlm" in model_id:
            first = list(samples)[0]
            print("   ", tk.encode(samples[first], add_special_tokens=False).tokens)
except Exception as e:
    print("Не удалось загрузить токенизатор (нужен интернет в Colab):", e)

## 5. Задания

1. Добавьте в `STEPS` шаг «числа → `<NUM>`» (замена, а не удаление) сразу после удаления пунктуации, иначе угловые скобки будут удалены. Как изменились числа токенов и типов? Почему такая замена бывает лучше удаления?
2. Найдите в разделе 2 тип, который «склеился» неправильно (две разные по смыслу формы стали одной строкой) или, наоборот, не склеился (одно слово в двух формах). Какой шаг или какая тема курса это исправит?
3. Обучите BPE на `corpus`, где все частоты равны 1. Как изменились первые 10 слияний и разбиение слова «новейшая»? Объясните, зачем BPE учитывает частоты слов.
4. Напишите `segment_with_unk(word, merges, alphabet)`, которая заменяет символы вне алфавита на `[UNK]`. Проверьте её на `new_words` из раздела 3: в каких словах появился `[UNK]` и какая доля их токенов неизвестна?
5. По данным раздела 4 найдите наименьшее число слияний K, при котором фертильность на тестовых словах в полтора раза меньше, чем при K = 0. Сколько при этом символов в словаре? Почему тестовая кривая всегда выше обучающей?

In [ ]:
# Ваш код здесь

In [ ]:
# Ваш код здесь

In [ ]:
# Ваш код здесь

In [ ]:
# Ваш код здесь

In [ ]:
# Ваш код здесь